In [21]:
def transformer_accounting_mem_params(config_dict: dict[str, int], precision: str = "fp32", verbose: bool = False):
    """
    Calculate the total number of parameters in a transformer model based on the configuration dictionary.

    Args:
        config_dict (dict): Dictionary containing model configuration parameters.
        precision (str): Precision type, currently not used but can be extended for different calculations.
    """
    precision_dict = {
        "fp32": 32,
        "fp16": 16,
        "int8": 8,
        "int4": 4}

    if precision not in precision_dict:
        raise ValueError(f"Currently only {list(precision_dict.keys())} precision is supported.")
    if not config_dict:
        raise ValueError("Configuration dictionary cannot be empty.")

    vocab_size = config_dict["vocab_size"]
    d_model = config_dict["d_model"]
    d_ff = config_dict["d_ff"]
    n_layers = config_dict["num_layers"]

    # input embedding param count
    input_emb = vocab_size * d_model

    # each transformer block
    ln1 = d_model
    qkv_proj = 3 * d_model * d_model
    out_proj = d_model * d_model
    ln2 = d_model
    ff1 = d_model * d_ff
    ff2 = d_ff * d_model
    transformer_block = ln1 + qkv_proj + out_proj + ln2 + ff1 + ff2
    transformer_blocks = n_layers * transformer_block

    # output norm and embedding
    output_norm = d_model
    output_emb = vocab_size * d_model

    total_params = input_emb + transformer_blocks + output_norm + output_emb
    total_memory = total_params * precision_dict[precision] / (8 * 1024 ** 3)  # in GB

    if verbose:
        print(f"Total parameters in transformer with {precision} precision: "
              f"{total_params:,} parameters, "
              f"{total_memory:.2f} GB")
    
    return total_params, total_memory

def transformer_accounting_flops(config_dict: dict[str, int], verbose: bool = False) -> float | None:
    """
    Calculate the total number of FLOPs in a transformer model based on the configuration dictionary.

    Args:
        config_dict (dict): Dictionary containing model configuration parameters.
        precision (str): Precision type, currently not used but can be extended for different calculations.
    """

    if not config_dict:
        return
    
    context_len = config_dict.get("context_len", 1024)  # Default context length if not specified
    vocab_size = config_dict["vocab_size"]
    d_model = config_dict["d_model"]
    d_ff = config_dict["d_ff"]
    n_layers = config_dict["num_layers"]
    d_ff_mode = config_dict.get("d_ff_mode", "linear")  # Default to linear if not specified

    # feed forward in transformer block
    # each matric mulitply for m * n and n * p matrix is 2mnp FLOPS

    qkv_proj_flops = 2 * d_model * context_len * 3 * d_model # (context_len, d_model) @ (d_model, 3 * d_model)
    qkt_flops = 2 * d_model * context_len**2 # (context_len, d_model) @ (d_model, context_len)
    qkt_v_flops = 2 * d_model * context_len**2 # (context_len, context_len) @ (context_len, d_model)
    out_proj_flops = 2 * context_len * d_model**2 # (context_len, d_model) @ (d_model, d_model)
    attn_block_flops = qkv_proj_flops + qkt_flops + qkt_v_flops + out_proj_flops

    if d_ff_mode == "swiglu":
        # feed forward in transformer block with swiglu activation
        ff1_flops = 4 * d_model * context_len * d_ff # (context_len, d_model) @ (d_model, d_ff) 2 linear projections
        # and 2 swiglu activations
        ff2_flops = 2 * d_ff * context_len * d_model # (context_len, d_ff) @ (d_ff, d_model)
    else:    
        # feed forward in transformer block
        ff1_flops = 2 * d_model * context_len * d_ff # (context_len, d_model) @ (d_model, d_ff)
        ff2_flops = 2 * d_ff * context_len * d_model # (context_len, d_ff) @ (d_ff, d_model)

    transformer_block_flops = attn_block_flops + ff1_flops + ff2_flops
    transformer_blocks_flops = n_layers * transformer_block_flops

    # lm head flops
    output_emb_flops = vocab_size * d_model

    total_flops = transformer_blocks_flops + output_emb_flops

    print(f"Total FLOPs in transformer: {total_flops / 1e12:.2F} TFlops")
    
    if verbose:
        print(f"Total FLOPs in transformer with context length {context_len} and {d_ff_mode} d_ff mode: "
              f"Attention block: {attn_block_flops / 1e12:.2F} TFlops, "
              f"Feed forward block: {ff1_flops / 1e12:.2F} TFlops")
        print(f"Total FLOPs in transformer with {n_layers} layers: "
              f"{transformer_blocks_flops / 1e12:.2F} TFlops, "
              f"Output embedding: {output_emb_flops / 1e12:.2F} TFlops")
    return total_flops


In [29]:
def optimizer_accounting(config_dict: dict[str, int], precision: str = "fp32", batch_size =1, verbose: bool = False):
    """
    Calculate the total number of parameters in an optimizer based on the configuration dictionary.

    Args:
        config_dict (dict): Dictionary containing model configuration parameters.
        precision (str): Precision type, currently not used but can be extended for different calculations.
    """
    if not config_dict:
        raise ValueError("Configuration dictionary cannot be empty.")
    precision_dict = {
        "fp32": 32,
        "fp16": 16,
        "int8": 8,
        "int4": 4
        }
    if precision not in precision_dict:
        raise ValueError(f"Currently only {list(precision_dict.keys())} precision is supported. Use fp16 for bfloat16.")

    vocab_size = config_dict["vocab_size"]
    d_model = config_dict["d_model"]
    d_ff = config_dict["d_ff"]
    n_layers = config_dict["num_layers"]
    num_heads = config_dict["num_heads"]
    context_length = config_dict.get("context_len", 1024)  # Default context length if not specified

    # adamw optimizer
    # you need memory for parameters, gradients, moments, activations
    # each parameter has 2 moments and 1 gradient, so 4 * param memory
    total_params, _ = transformer_accounting_mem_params(config_dict, precision)
    total_params = total_params * 4  # 4 * parameters for adamw optimizer

    # activations
    rms_norm1 =  context_length * d_model  # for each layer
    qkv_proj =  context_length * d_model * 3  # for each layer
    attn_qtk =  num_heads * context_length**2
    attn_scores =  num_heads * context_length**2
    scores_values =  context_length * d_model  # for each layer
    out_proj =  context_length * d_model  # for each layer

    rms_norm2 =  context_length * d_model
    ff1 =  context_length * d_ff  # for each layer
    silu =  context_length * d_ff  # for each layer
    ff2 =  context_length * d_model  # for each layer

    transformer_block_activations = (rms_norm1 + qkv_proj + attn_qtk + attn_scores + scores_values + out_proj +
                                      rms_norm2 + ff1 + silu + ff2)
    total_tf_activations = n_layers * transformer_block_activations

    final_norm =  context_length * d_model  # final layer norm
    output_emb =  vocab_size * context_length
    crs_entropy =  context_length * vocab_size  # cross entropy loss

    total_activations = (total_tf_activations + final_norm + output_emb + crs_entropy)

    total_mem = (total_params + (batch_size * total_activations)) * precision_dict.get(precision, 32) / (8 * 1024 ** 3)  # in GB

    if verbose:
        print(f"Total memory for optimizer AdamW with {precision} precision: {total_mem:.2f} GB")

    return total_mem

In [30]:
gpt2_xl_config = {
    "vocab_size": 50257,
    "d_model": 1600,
    "d_ff": 6400,
    "num_layers": 48,
    "num_heads": 25,
    "context_len": 1024,
    "d_ff_mode": "linear"
}

peak_mem =  optimizer_accounting(gpt2_xl_config, precision="fp32", batch_size=1, verbose=True)
print("Max batch size for 80GB GPU:", int(80.0/peak_mem))

Total memory for optimizer AdamW with fp32 precision: 38.82 GB
Max batch size for 80GB GPU: 2


In [17]:
peak_mem

32.73059105873108